**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/zaimi-2018"], check=True)
    os.chdir("/content/encore/entries/zaimi-2018")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Zaimi et al. (2018), AxonDeepSeg, tested on dandiset 001436

**Paper.** Zaimi A, Wabartha M, Herman V, Antonsanti P-L, Perone CS, Cohen-Adad J (2018).
AxonDeepSeg: automatic axon and myelin segmentation from microscopy data using convolutional
neural networks. *Scientific Reports* 8, 3816. doi:10.1038/s41598-018-22181-4.

**Dataset.** DANDI:001436, "TEM Images of Corpus Callosum in Control and Cuprizone-Intoxicated
Mice with Axon and Myelin Segmentations", version **0.250512.1625**
(doi:10.48324/dandi.001436/0.250512.1625). The paper cites no version of the dandiset (the dandiset
was created in 2025, after the paper). 0.250512.1625 is the only published version, and the
draft (modified 2026-09-23) has the same 660 assets and size, so the most recent published
version is pinned. The dataset is a BIDS microscopy dataset of png images, not NWB; the files
are read from the archive into memory.

**Text the claims were written from.** The version of record (PMC5830647), with Methods,
figure legends and Tables 1 to 3; the paper has no supplement. Claims file:
`claims/zaimi-2018.yaml`.

**Outside inputs.** (1) The paper's released TEM model, `default_TEM_model_v1`, at commit
7f6013221b2a862e107a7e23f3c8662eba3ba64b of https://github.com/neuropoly/axondeepseg, with the
inference code at the same commit (claims `data: with_inputs`). (2) The folder names of the NYU
mouse component (OSF node `adrcs`) of the White Matter Microscopy Database
(https://osf.io/yp4qg/, doi:10.17605/OSF.IO/YP4QG), which the paper's Data availability
statement names, to label each mouse as control or cuprizone (`data: with_labels`, tested by
the link claim `osf-group-labels-link`). (3) The macaque TEM images and masks of ref. 7 in the
same database (OSF node `yp4qg`, folder MontrealNeurologicalInstitute/histology/Macaque), for
the two macaque claims (`data: elsewhere`). (4) The SEM dataset of AxonDeepSeg,
https://github.com/axondeepseg/data_axondeepseg_sem at tag r20211130 (commit d7884f9,
doi:10.5281/zenodo.5498378), whose rat images include two with the pixel size, field of view and
fixation of the paper's SEM test images, for table3-sem-rat-accuracy (`data: elsewhere`).

**Ground truth.** The dandiset's masks are not those the paper scored against: its CHANGES file
(printed below) says that in 2025 the masks of 155 of the 158 images were changed "to separate all
fibers with a 2px delineation", and one mask was corrected in 2022. The earlier masks are not
public. A variant that fills the delineation lines bounds the effect on the mouse scores.

| Claim | Tested here |
|---|---|
| osf-group-labels-link (added by this attempt) | tested |
| table3-tem-mice-accuracy | tested |
| table3-tem-mice-dice-detection | tested |
| myelin-dice-below-axon-dice | tested (mouse images only) |
| sensitivity-above-93 | tested (mouse images only) |
| table3-sem-rat-accuracy | tested on the SEM dataset of AxonDeepSeg (data elsewhere) |
| table3-sem-human-accuracy | blocked, the human SEM test image is not public |
| table3-tem-macaque-accuracy | tested on the macaque images of the White Matter Microscopy Database (data elsewhere) |
| table3-other-samples-dice-detection | blocked, the human SEM test image is not public (rat and macaque values shown) |
| macaque-myelin-underestimated | tested on the macaque images of the White Matter Microscopy Database (data elsewhere) |
| full-slice-processing-time | blocked, no SEM slice in the dandiset |
| fig7-corticospinal-tract | blocked, no rat spinal cord in the dandiset |
| fig7-larger-axons-periphery | blocked, as above |
| fig7-tract-diameters | blocked, as above |
| fig7-gratio-range | blocked, as above |

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import itertools
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import axondeepseg_tem as ads

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("dandiset", ads.DANDISET_ID, "version", ads.DANDISET_VERSION)
print("AxonDeepSeg commit", ads.ADS_COMMIT)

## Data

The archive's metadata, the file list and the sidecar files. Each mouse has one sidecar
(`sub-<mouse>_TEM.json`) giving the pixel size.

In [ ]:
meta = ads.version_metadata()
print(meta["name"])
print("assets", meta["assetsSummary"]["numberOfFiles"], "bytes", meta["assetsSummary"]["numberOfBytes"],
      "subjects", meta["assetsSummary"]["numberOfSubjects"], "samples (archive summary)",
      meta["assetsSummary"]["numberOfSamples"])
print(meta["description"])

assets = ads.list_assets()
kind = assets.path.str.replace(r"sub-nyuMouse\d+", "sub-*", regex=True).str.replace(r"sample-\d+", "sample-*", regex=True)
print(kind.value_counts().to_string())
for p in ["README", "CHANGES", "participants.tsv", "samples.json", "derivatives/labels/dataset_description.json"]:
    print("=====", p)
    print(ads.read_bytes(assets.set_index("path").asset_id[p]).decode())

In [ ]:
table = ads.sample_table(assets)
sidecars = {s: json.loads(ads.read_bytes(a)) for s, a in table.groupby("subject").sidecar.first().items()}
print(pd.DataFrame(sidecars).T.astype(str).drop_duplicates().to_string())
table["pixel_um"] = table.subject.map({s: d["PixelSize"][0] for s, d in sidecars.items()})
samples_tsv = pd.read_csv(pd.io.common.BytesIO(ads.read_bytes(assets.set_index("path").asset_id["samples.tsv"])), sep="\t")
listed = set(zip(samples_tsv.participant_id, samples_tsv.sample_id))
found = set(zip(table.subject, table["sample"]))
print("images", len(table), "with all three masks", int(table[["axon", "myelin", "axonmyelin"]].notna().all(axis=1).sum()))
print("samples.tsv rows", len(samples_tsv), "| listed but no image", sorted(listed - found), "| image not listed", sorted(found - listed))
print("images per mouse:", table.groupby("subject").size().to_dict())

### Group labels from the White Matter Microscopy Database

The dandiset does not say which mice are controls and which received cuprizone, though its
description says it holds both. The claims name, as one reading of the test set, the control
mice only. The paper's Data availability statement names the White Matter Microscopy Database;
its NYU component (OSF node `adrcs`) holds the raw micrographs of ref. 27 in one folder per diet
group and time point, with one sub-folder per mouse named by the mouse's number. Only the folder
names are used.

In [ ]:
osf = ads.osf_mouse_folders()
print(osf[["group_folder", "mouse_folder", "mouse"]].to_string())

## The model

The paper's TEM model is read from GitHub at the pinned commit. Its network is rebuilt from the
checkpoint (see the helper's docstring), and checked here against the model's own saved
TensorFlow graph (`model.ckpt.meta`), run with `training_phase` False, on the four patches of the
first image.

In [ ]:
weights, config = ads.load_tem_model()
print("variables", len(weights), "parameters", sum(int(np.prod(v.shape)) for v in weights.values()))
print({k: config[k] for k in ["depth", "trainingset_patchsize", "n_classes", "convolution_per_layer", "downsampling", "batch_norm_activate"]})

first = table.iloc[0]
img0, gt0, _ = ads.load_sample(first)
from skimage.transform import rescale
rs0 = rescale((255 - img0.astype(int)).astype(np.uint8), first.pixel_um / ads.WORKING_PIXEL_UM, order=1,
              mode="constant", preserve_range=True, anti_aliasing=False).astype(int)
patches0, pos0 = ads.im2patches_overlap(rs0)
mine = ads.unet_forward(weights, config, patches0)
theirs = ads.original_graph_output(patches0)
print("patches", len(patches0), "positions", pos0)
print("largest absolute difference of the outputs", float(np.abs(mine - theirs).max()),
      "| output range", float(theirs.min()), float(theirs.max()))
print("fraction of pixels with the same class", float((mine.argmax(3) == theirs.argmax(3)).mean()))

### The inference steps of the original code

The parts of the pinned code that the helper repeats outside the network, printed so that they can
be compared with `segment` and `resize_back`: the contrast inversion for this model (`segment.py`),
the working pixel sizes, the resampling of the input, the preprocessing of each patch, and the resize
of the class map back to the native shape followed by the cast to uint8 (`apply_model.py`). The code
ran with scikit-image 0.13 (it required >= 0.12.3); the lines of scikit-image 0.13.1 that set the
defaults of `resize` and `rescale` (order 1, mode 'constant' when none is given) and the exact integer
histogram of `equalize_hist` are printed too, as these defaults changed later. The original pipeline
(Python 2, TensorFlow 1) cannot be installed here, so its output is not compared end to end.

In [ ]:
for path, a, b in [("AxonDeepSeg/segment.py", 149, 154), ("AxonDeepSeg/segment.py", 239, 249),
                   ("AxonDeepSeg/apply_model.py", 332, 335), ("AxonDeepSeg/patch_management_tools.py", 5, 16),
                   ("AxonDeepSeg/apply_model.py", 416, 421)]:
    print(f"----- {path}, lines {a} to {b}, at {ads.ADS_COMMIT[:7]}")
    print(ads.code_lines(path, a, b))
for path, a, b in [("skimage/transform/_warps.py", 21, 22), ("skimage/transform/_warps.py", 82, 83),
                   ("skimage/transform/_warps.py", 140, 141), ("skimage/exposure/exposure.py", 67, 86)]:
    print(f"----- scikit-image v0.13.1, {path}, lines {a} to {b}")
    print(ads.code_lines(path, a, b, commit="v0.13.1", repo="scikit-image/scikit-image"))
# The cast after the bilinear resize truncates: a pixel halfway between axon (2) and background (0) becomes myelin (1).
demo = np.array([[2.0, 0.0]])
print("resize_back of [[2, 0]] to 1 x 3:", ads.resize_back(demo, (1, 3)).tolist(), "| nearest neighbour:", ads.resize_back(demo, (1, 3), order=0).tolist())


## Segmenting every image

Each image is segmented as `segment.py` does for this model (contrast inverted once, 0.01 um
working pixel size, 512 x 512 patches with an overlap of 25 pixels), and, as the variant the
criterion asks for, without the inversion. Scores are computed against the dandiset's masks at
the native pixel size.

In [ ]:
t0 = time.time()
res = ads.run_all(table, weights, config)
res["mouse"] = res.subject.str.extract(r"(\d+)$")[0].astype(int)
print(f"{len(res)} images segmented twice in {(time.time() - t0) / 60:.1f} min")

## Data check

The masks are checked against their images below. They are the 2025 revision (CHANGES, printed
above), in which touching fibres are separated by a 2-pixel line of background; the masks of 2018
are not in the dandiset, which has one published version.


In [ ]:
print("images", len(res), "mice", res.subject.nunique())
print("images per mouse:", res.groupby("subject").size().to_dict())
print("pixel size (um) in the sidecars:", sorted(table.pixel_um.unique()), "| Table 1 of the paper gives 0.002")
print("image shapes:", res.groupby(["height", "width"]).size().to_dict())
print("field of view (um): %.2f x %.2f" % (res.height.iloc[0] * table.pixel_um.iloc[0], res.width.iloc[0] * table.pixel_um.iloc[0]),
      "| Table 1 gives 6 x 9")
print("grey levels: min over images", res.img_min.min(), "max", res.img_max.max(), "| lowest image max", res.img_max.min())
print("axon mask values:", res.axon_values.value_counts().to_dict())
print("myelin mask values:", res.myelin_values.value_counts().to_dict())
print("axonmyelin mask values:", res.axonmyelin_values.value_counts().to_dict())
print("dimensions of the axon mask, myelin mask and image:", res.mask_dims.value_counts().to_dict())
print("axonmyelin pngs with 1 channel (grey) and 2 (grey and alpha):", res.axonmyelin_channels.value_counts().to_dict(),
      "| alpha values:", res.axonmyelin_alpha.dropna().value_counts().to_dict())
print("axonmyelin mask (grey channel) equals axon (255) and myelin (128) masks in", int(res.am_agrees.sum()), "of", len(res), "images")
print("pixels both axon and myelin: total", int(res.axon_myelin_overlap.sum()))
print("ground-truth fractions: axon %.3f to %.3f, myelin %.3f to %.3f" % (res.inv_gt_axon_frac.min(), res.inv_gt_axon_frac.max(),
      res.inv_gt_myelin_frac.min(), res.inv_gt_myelin_frac.max()))
print("ground-truth axon objects per image: %d to %d" % (res.inv_P.min(), res.inv_P.max()))

# Masks in register with their images: myelin is the darkest structure in TEM.
print("grey-level contrast, rest of image minus myelin: masks in place median %.1f (min %.1f); "
      "shifted by 50 px median %.1f (max %.1f); larger in place in %d of %d images" % (
      res.contrast.median(), res.contrast.min(), res.contrast_shifted.median(), res.contrast_shifted.max(),
      int((res.contrast > res.contrast_shifted).sum()), len(res)))

# Images deposited twice.
print("distinct images by content hash:", res.sha1.nunique(), "| distinct ground truths:", res.gt_sha1.nunique())
T = np.asarray(list(res.thumb), float)
T = (T - T.mean(1, keepdims=True)) / T.std(1, keepdims=True)
R = T @ T.T / T.shape[1]
iu = np.triu_indices(len(res), 1)
print("largest thumbnail correlation between two different images: %.3f; pairs above 0.95: %d" % (
      R[iu].max(), int((R[iu] > 0.95).sum())))

# The pixel size of the paper (0.002 um) would give images too small for a 512 x 512 patch.
h002 = np.round(res.height * 0.002 / ads.WORKING_PIXEL_UM).astype(int)
print("height after resampling with 0.002 um: %d to %d px (a patch is 512); with the sidecar's %.5f um: %s" % (
      h002.min(), h002.max(), table.pixel_um.iloc[0], sorted({s[0] for s in res.resampled_shape})))
print("patches per image:", res.n_patches.value_counts().to_dict())

## osf-group-labels-link

> Each mouse of the dandiset is found, by its number, in exactly one diet-group folder of the
> NYU component of the White Matter Microscopy Database, and the deposits agree on what both
> hold (added by this attempt; see the claims file for the criterion).

In [ ]:
grp = osf.groupby("mouse").group_folder.agg(lambda s: sorted(set(s)))
dmice = sorted(res.mouse.unique())
link = pd.DataFrame({"mouse": dmice})
link["osf_groups"] = link.mouse.map(lambda m: grp.get(m, []))
link["n_groups"] = link.osf_groups.map(len)
link["group"] = link.osf_groups.map(lambda g: g[0] if len(g) == 1 else None)
link["control"] = link.group.str.startswith("ctl")
print(link.to_string())
print("dandiset mice matched to exactly one group folder:", int((link.n_groups == 1).sum()), "of", len(link))
print("mice per group folder in the dandiset:", link.group.value_counts().to_dict())
print("mice per group folder on OSF:", osf.groupby("group_folder").mouse.nunique().to_dict())
print("controls:", list(link.mouse[link.control]), "| cuprizone:", list(link.mouse[~link.control]))

In [ ]:
# Content: for each matched mouse whose OSF folder holds full-size micrographs, up to four of them
# (in name order, max_files=4 in osf_thumbnails) are compared with every dandiset image by thumbnail correlation.
shape = (int(res.height.iloc[0]), int(res.width.iloc[0]))
rows = []
for _, fr in osf[osf.mouse.isin(dmice)].iterrows():
    files = ads.osf_files(fr.files_url)
    modified = dict(zip(files.id, files.modified))
    for name, fid, oshape, th in ads.osf_thumbnails(files, shape):
        if th is None:
            rows.append({"osf_folder": fr.mouse_folder, "file": name, "shape": oshape, "note": "smaller than the dandiset images"})
            continue
        b = ads.best_match(th, list(res.thumb))
        i = int(np.argmax(b))
        same = b[(res.mouse == fr.mouse).values]
        other = b[(res.mouse != fr.mouse).values]
        rows.append({"osf_folder": fr.mouse_folder, "file": name, "id": fid, "modified": str(modified.get(fid))[:10], "shape": oshape,
                     "best_image": f"{res.subject[i]} {res['sample'][i]}", "r_best": round(float(b[i]), 3),
                     "best_same_mouse_r": round(float(same.max()), 3), "best_other_mouse_r": round(float(other.max()), 3)})
content = pd.DataFrame(rows)
print(content.to_string())
ok = content.dropna(subset=["r_best"]).copy()
ok["mouse"] = ok.osf_folder.str.extract(r"^(\d+)")[0].astype(int)
ok["best_mouse"] = ok.best_image.str.extract(r"nyuMouse(\d+)")[0].astype(int)
cp = ok[ok.r_best > 0.5]
print("OSF micrographs compared:", len(ok), "from", ok.mouse.nunique(), "mice | with a counterpart (r > 0.5):", len(cp),
      "| counterpart of the same mouse:", int((cp.mouse == cp.best_mouse).sum()),
      "| of another mouse:", int((cp.mouse != cp.best_mouse).sum()))
print("mice with at least one counterpart:", sorted(cp.mouse.unique()), "| mice checked without one:",
      sorted(set(ok.mouse) - set(cp.mouse)))
print("r of counterparts: %.3f to %.3f; largest r of the others: %.3f" % (
      cp.r_best.min(), cp.r_best.max(), ok[ok.r_best <= 0.5].r_best.max() if (ok.r_best <= 0.5).any() else float("nan")))
print("modification dates of the compared OSF micrographs:", sorted(ok.modified.unique()))
print("matched dandiset mice whose OSF folders hold no full-size micrograph:",
      sorted(set(dmice) - set(ok.mouse)))

**Result: reproduced.** All 20 dandiset mice are matched by number to exactly one group folder of
the NYU component of the White Matter Microscopy Database, and no group folder holds more than 3
dandiset mice (the control group at 6 weeks holds 2: mouse 17 is not in the dandiset). Of 39 OSF
micrographs compared, 16 have a counterpart in the dandiset (r from 0.593 to 0.903), every one an
image of the same mouse, from 8 of the 11 mice checked; the others correlate at most 0.487 with any
dandiset image, as expected for the OSF images the dandiset does not hold. Nine dandiset mice
(12 and 29 to 36) have no full-size micrograph on OSF, so they are linked by number alone. The
link gives 8 control mice (7, 9, 25, 26, 27, 34, 35, 36) and 12 cuprizone mice.

## Per-image scores, and the test-set readings

The paper's test set is 8 images from each of 3 mice not used for training (Table 1). The
dandiset holds the 20 mice of the paper's training and test sets together, without saying which
3 were tested. The claims name two readings: (a) every image of the dandiset, and (b) the
images of the control mice only. The results for the cuprizone mice apart, per mouse, and, as a
variant, the range over every possible set of 3 mice (all 1140), are shown too.

In [ ]:
res["control"] = res.mouse.map(dict(zip(link.mouse, link.control)))
res["group"] = res.mouse.map(dict(zip(link.mouse, link.group)))
M = ["axon_dice", "myelin_dice", "accuracy", "sensitivity", "precision"]
EXPECTED = {"accuracy": (0.9451, 0.0945), "axon_dice": (0.9493, 0.0949), "myelin_dice": (0.8552, 0.0855),
            "sensitivity": (0.9597, 0.0960), "precision": (0.9647, 0.0965)}
readings = {"(a) all images": res, "(b) control mice": res[res.control], "cuprizone mice (not a reading)": res[~res.control]}

def summary(tag):
    rows = {}
    for name, d in readings.items():
        rows[name] = {"images": len(d), "mice": d.mouse.nunique(), **{m: d[f"{tag}_{m}"].mean() for m in M}}
    return pd.DataFrame(rows).T

per_mouse = res.groupby(["mouse", "group"])[[f"inv_{m}" for m in M]].mean().round(4)
per_mouse.columns = M
print("Per mouse, contrast inverted (segment.py):")
print(per_mouse.to_string())
mouse_means = res.groupby("mouse")[[f"inv_{m}" for m in M]].mean()
mouse_n = res.groupby("mouse").size()
combos = list(itertools.combinations(mouse_means.index, 3))
sub = {m: [] for m in M}
for c in combos:
    w = mouse_n[list(c)].values
    for m in M:
        sub[m].append(np.average(mouse_means.loc[list(c), f"inv_{m}"].values, weights=w))
ctl_mice = list(link.mouse[link.control])
ctl_combos = [c for c in combos if set(c) <= set(ctl_mice)]
print(len(combos), "sets of 3 mice;", len(ctl_combos), "of them all controls")

## table3-tem-mice-accuracy

> The TEM model trained on mouse images segments the mouse test images with a mean three-class
> pixel-wise accuracy of 95% (0.9451 in Table 3, mean over 24 test images).

Expected 0.9451, tolerance 0.0945 (0.8506 to 1.0396).

In [ ]:
lo, hi = EXPECTED["accuracy"][0] - EXPECTED["accuracy"][1], EXPECTED["accuracy"][0] + EXPECTED["accuracy"][1]
s_inv, s_no = summary("inv"), summary("noinv")
print("Contrast inverted once (segment.py):")
print(s_inv[["images", "mice", "accuracy"]].round(4).to_string())
print("Variant, no inversion:")
print(s_no[["images", "mice", "accuracy"]].round(4).to_string())
for name in ["(a) all images", "(b) control mice"]:
    v = s_inv.loc[name, "accuracy"]
    print(f"{name}: {v:.4f}, within {lo:.4f} to {hi:.4f}: {lo <= v <= hi}")
print("per-image accuracy: %.4f to %.4f (median %.4f); per-mouse means %.4f to %.4f" % (
      res.inv_accuracy.min(), res.inv_accuracy.max(), res.inv_accuracy.median(),
      per_mouse.accuracy.min(), per_mouse.accuracy.max()))
print("over all 1140 sets of 3 mice: %.4f to %.4f; over the %d sets of 3 control mice: %.4f to %.4f" % (
      min(sub["accuracy"]), max(sub["accuracy"]), len(ctl_combos),
      min(sub["accuracy"][combos.index(c)] for c in ctl_combos), max(sub["accuracy"][combos.index(c)] for c in ctl_combos)))
print("images below the lower bound %.4f: %d of %d" % (lo, int((res.inv_accuracy < lo).sum()), len(res)))
print("Variant, ground truth with the 2025 delineation lines filled (fill_delineation): pixels changed per image "
      "%.4f to %.4f (median %.4f); images with no pixel changed: %d" % (
      res.fill_frac.min(), res.fill_frac.max(), res.fill_frac.median(), int((res.fill_frac == 0).sum())))
for name in ["(a) all images", "(b) control mice"]:
    d = readings[name]
    print(f"  {name}: accuracy {d.fill_accuracy.mean():.4f} (against {d.inv_accuracy.mean():.4f} with the dandiset's masks)")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
order = per_mouse.sort_values("accuracy").index
for i, (key, d) in enumerate(res.groupby("mouse")):
    ax[0].scatter(np.full(len(d), list(order.get_level_values(0)).index(key)), d.inv_accuracy,
                  c="tab:blue" if d.control.iloc[0] else "tab:orange", s=12)
ax[0].axhspan(lo, min(hi, 1.0), color="0.9", zorder=0)
ax[0].axhline(0.9451, color="k", lw=0.8)
ax[0].set_xticks(range(len(order)), [str(m) for m in order.get_level_values(0)], fontsize=7)
ax[0].set_xlabel("mouse (blue control, orange cuprizone)"); ax[0].set_ylabel("pixel-wise accuracy")
ax[0].set_ylim(0.85, 1.0)
ax[1].hist(sub["accuracy"], bins=30, color="0.5")
ax[1].axvline(0.9451, color="k", lw=0.8)
ax[1].set_xlabel("mean accuracy, every set of 3 mice"); ax[1].set_ylabel("sets")
plt.tight_layout(); plt.show()
print("left panel: %d images, accuracy %.4f to %.4f; points below 0.85 (outside the axis): %d" % (
      len(res), res.inv_accuracy.min(), res.inv_accuracy.max(), int((res.inv_accuracy < 0.85).sum())))
print("right panel: %d sets, %.4f to %.4f" % (len(sub["accuracy"]), min(sub["accuracy"]), max(sub["accuracy"])))

**Result: reproduced.** The mean pixel-wise accuracy
is 0.9416 over all 158 images (reading a) and 0.9349 over the 62 images of the 8 control mice
(reading b), both inside 0.8506 to 1.0396; the cuprizone mice give 0.9459. Every set of 3 mice,
which includes the paper's unnamed test set, gives 0.9226 to 0.9683, and no single image is below
the lower bound (lowest 0.8954), so the outcome does not depend on which mice the paper
tested. Both readings include images of the mice the model was trained
on, so the match is weaker evidence than a test on the paper's held-out mice. If the dandiset's 20
mice are the paper's 20, 12 of them cuprizone mice, then the paper's training set held cuprizone
tissue, which bears on reading (b) and its premise of healthy tissue; this does not change the
outcome, as every set of 3 mice passes. With the 2025 delineation lines filled in the ground truth
(a variant for the masks the paper scored against, which are not public) the accuracy is 0.9427 (a)
and 0.9366 (b), within 0.002 of the values with the dandiset's masks. Without the contrast
inversion the accuracy falls to 0.5027 (a) and 0.3934 (b): the inversion that `segment.py`
applies for this model is needed.

## table3-tem-mice-dice-detection

> On the mouse test images the TEM model reaches a mean axon Dice of 0.9493, a mean myelin Dice
> of 0.8552, a mean axon detection sensitivity of 0.9597 and a mean precision of 0.9647.

Tolerance 10% of each value.

In [ ]:
q = ["axon_dice", "myelin_dice", "sensitivity", "precision"]
for name in ["(a) all images", "(b) control mice", "cuprizone mice (not a reading)"]:
    within = []
    for m in q:
        v = s_inv.loc[name, m]; e, t = EXPECTED[m]
        within.append(abs(v - e) <= t)
        print(f"{name:32s} {m:12s} {v:.4f}  expected {e} +/- {t}  within: {abs(v - e) <= t}")
    print(f"{name}: {sum(within)} of 4 within tolerance")
print("Variant, no inversion:")
print(s_no[q].round(4).to_string())
print("per-image ranges:", {m: (round(res[f'inv_{m}'].min(), 4), round(res[f'inv_{m}'].max(), 4)) for m in q})
print("per-mouse ranges:", {m: (round(per_mouse[m].min(), 4), round(per_mouse[m].max(), 4)) for m in q})
print("over all 1140 sets of 3 mice:", {m: (round(min(sub[m]), 4), round(max(sub[m]), 4)) for m in q})
for name, d in readings.items():
    print(f"{name}: ground-truth axons {int(d.inv_P.sum())}, detected (TP) {int(d.inv_TP.sum())}, false positives {int(d.inv_FP.sum())}")
print("Variant, ground truth with the 2025 delineation lines filled: mean myelin Dice",
      {name: round(readings[name].fill_myelin_dice.mean(), 4) for name in ["(a) all images", "(b) control mice"]},
      "| axon Dice, sensitivity and precision are unchanged, as the filled pixels become myelin")


**Result: reproduced.** Under reading (a) the means are axon Dice
0.9500, myelin Dice 0.8489, sensitivity 0.9863 and precision 0.9707; under reading (b) 0.9543,
0.8571, 0.9878 and 0.9819. All four are within tolerance under both readings, and over every set
of 3 mice each measure stays within its tolerance (axon Dice 0.9300 to 0.9651, myelin Dice
0.8175 to 0.8739, sensitivity 0.9722 to 0.9969, precision 0.9304 to 0.9949). Sensitivity and
precision are above the paper's values under both readings. Without the inversion all four
collapse (axon Dice 0.0029). With the 2025 delineation lines filled in the ground truth the mean
myelin Dice is 0.8521 (a) and 0.8611 (b); the other three measures do not change.

## myelin-dice-below-axon-dice

> In both models and every test sample the myelin Dice is lower than the axon Dice, which the
> authors attribute to myelin having two boundaries (with the axon and with the background).

Tested on the mouse images only: (i) the means of a test set, (ii) every single image.

In [ ]:
for name in ["(a) all images", "(b) control mice"]:
    d = readings[name]
    below = d.inv_myelin_dice < d.inv_axon_dice
    print(f"{name}: (i) mean myelin Dice {d.inv_myelin_dice.mean():.4f} < mean axon Dice {d.inv_axon_dice.mean():.4f}: "
          f"{d.inv_myelin_dice.mean() < d.inv_axon_dice.mean()}; (ii) single images with myelin below axon: "
          f"{int(below.sum())} of {len(d)}")
diff = res.inv_axon_dice - res.inv_myelin_dice
print("axon minus myelin Dice per image: %.4f to %.4f, median %.4f" % (diff.min(), diff.max(), diff.median()))
print("per mouse, myelin below axon in", int((per_mouse.myelin_dice < per_mouse.axon_dice).sum()), "of", len(per_mouse), "mice")

**Result: reproduced on the mouse images.** Myelin Dice is below axon Dice in the means of both
readings (0.8489 against 0.9500, and 0.8571 against 0.9543) and in every one of the 158 single
images (the difference ranges from 0.0655 to 0.1556). The SEM rows and the macaque row of
Table 3 are not part of this test.

## sensitivity-above-93

> Axon detection sensitivity is above 93% in every test set.

Tested on the mouse images only, with the mean sensitivity under each test-set reading.

In [ ]:
for name in ["(a) all images", "(b) control mice"]:
    v = readings[name].inv_sensitivity.mean()
    print(f"{name}: mean sensitivity {v:.4f}, above 0.93: {v > 0.93}")
print("images with sensitivity at or below 0.93: %d of %d (lowest %.3f)" % (
      int((res.inv_sensitivity <= 0.93).sum()), len(res), res.inv_sensitivity.min()))
print("lowest per-mouse mean %.4f; lowest over all 1140 sets of 3 mice %.4f" % (per_mouse.sensitivity.min(), min(sub["sensitivity"])))

**Result: reproduced on the mouse images.** The mean sensitivity is 0.9863 under reading (a) and
0.9878 under reading (b), above 0.93; the lowest per-mouse mean is 0.9682 and the lowest over
every set of 3 mice 0.9722. Two of the 158 single images are at or below 0.93 (lowest 0.898).
The SEM rows and the macaque row are not part of this test.

## The SEM rat test images (data elsewhere)

The SEM test images are not in the dandiset. The AxonDeepSeg project publishes an SEM dataset,
https://github.com/axondeepseg/data_axondeepseg_sem (DOI 10.5281/zenodo.5498378 in its
`dataset_description.json`, whose authors include the paper's authors): 10 rat cervical spinal
cord SEM images, "small cropped sections extracted from larger mosaic images" acquired from 2015 to
2017, with manual axon, myelin and axonmyelin masks. It is read here at commit d7884f9 (tag r20211130, the release archived
under that DOI), the last before the masks of 6 samples were changed in 2025 to separate touching fibres; the 2025
masks are a variant. Each image is matched to the rows of Table 1 by pixel size, field of view and
fixation. The images that match the two SEM test rows are segmented with the paper's
`default_SEM_model_v1` as `segment.py` does (0.1 um working pixel size for a 512 patch,
`generate_resolution`; overlap 25; no contrast inversion, which `segment.py` applies only for
`default_TEM_model_v1`).

In [ ]:
sem_list = ads.sem_list()
print(sem_list.to_string())
# Table 1 of the paper, SEM rows: (role, number of images, pixel size in um, field of view in um2, fixation PFA-GA %)
TABLE1_SEM = [("training", 1, "0.18", "230 x 166", "4-2"), ("training", 3, "0.05 to 0.17", "132 x 90 to 218 x 162", "4-0"),
              ("training", 3, "0.1", "74 x 76 to 77 x 84", "3-3"), ("training", 1, "0.13", "247 x 234", "3-3"),
              ("training", 1, "0.1", "82 x 77", "3-3"), ("test Rat 1", 1, "0.13", "150 x 97", "3-3"),
              ("test Rat 2", 1, "0.07", "108 x 77", "3-3"), ("test Human", 1, "0.13", "715 x 735", "4-2")]
print(pd.DataFrame(TABLE1_SEM, columns=["Table 1 row", "images", "pixel_um", "fov_um2", "fixation"]).to_string())
for name, (subj, samp) in ads.SEM_TEST.items():
    r = sem_list[(sem_list.subject == subj) & (sem_list["sample"] == samp)].iloc[0]
    print(f"{name}: {subj} {samp}, {r.pixel_um} um, {r.width_px} x {r.height_px} px = {r.fov_um} um, fixation {r.fixation}")

sem_weights, sem_config = ads.load_tem_model(ads.SEM_MODEL_DIR)
print("SEM model: variables", len(sem_weights), {k: sem_config[k] for k in ["depth", "trainingset_patchsize", "n_classes", "convolution_per_layer"]})
sem_rows, sem_conf = [], {}
for name, (subj, samp) in ads.SEM_TEST.items():
    img, gt, masks, px, raw = ads.load_sem(subj, samp)
    print(f"{name}: image {raw.shape} {raw.dtype}, grey {img.min()} to {img.max()}" +
          (f", alpha values {np.unique(raw[..., 1]).tolist()}" if raw.ndim == 3 else "") +
          f"; mask values axon {np.unique(masks['axon']).tolist()}, myelin {np.unique(masks['myelin']).tolist()}, "
          f"axonmyelin {np.unique(masks['axonmyelin']).tolist()}; axonmyelin agrees with axon and myelin masks: "
          f"{bool(np.array_equal(masks['axonmyelin'] == 255, masks['axon'] > 127) and np.array_equal(masks['axonmyelin'] == 127, masks['myelin'] > 127))}; "
          f"class fractions bg, myelin, axon {np.round(np.bincount(gt.ravel(), minlength=3) / gt.size, 3).tolist()}")
    if name == "Rat 1":
        rs = rescale(img, px / ads.SEM_WORKING_PIXEL_UM, order=1, mode="constant", preserve_range=True,
                     anti_aliasing=False).astype(int)
        p_, pos_ = ads.im2patches_overlap(rs)
        mine_s, theirs_s = ads.unet_forward(sem_weights, sem_config, p_), ads.original_graph_output(p_, ads.SEM_MODEL_DIR)
        print(f"SEM network against its saved graph on the {len(p_)} patches of Rat 1: largest absolute difference "
              f"{float(np.abs(mine_s - theirs_s).max()):.2g}, same class on {float((mine_s.argmax(3) == theirs_s.argmax(3)).mean()):.4f} of pixels")
    stitched, rs_shape, n_patches = ads.segment_stitched(img, px, sem_weights, sem_config, invert=False,
                                                         working_um=ads.SEM_WORKING_PIXEL_UM)
    pred = ads.resize_back(stitched, img.shape)
    sem_conf[name] = ads.confusion(pred, gt)
    variants = {"segment.py": (pred, gt),
                "nearest-neighbour resize back": (ads.resize_back(stitched, img.shape, order=0), gt),
                "contrast inverted": (ads.segment(img, px, sem_weights, sem_config, invert=True,
                                                  working_um=ads.SEM_WORKING_PIXEL_UM)[0], gt)}
    img25, gt25, _, _, _ = ads.load_sem(subj, samp, mask_commit=ads.SEM_COMMIT_2025)
    print(f"{name}: image the same at both commits: {bool(np.array_equal(img25, img))}; 2025 masks differ on "
          f"{float(np.mean(gt25 != gt)):.4f} of pixels")
    if not np.array_equal(gt25, gt):
        variants["2025 masks"] = (pred, gt25)
    for v, (pv, gv) in variants.items():
        sem_rows.append({"sample": name, "variant": v, "resampled": rs_shape, "patches": n_patches, **ads.scores(pv, gv)})
sem = pd.DataFrame(sem_rows)
print(sem.drop(columns=["pred_axon_frac", "gt_axon_frac"]).round(4).to_string())


## table3-sem-rat-accuracy

> The SEM model trained on rat spinal cord reaches a pixel-wise accuracy between 85% and 88% on the
> two rat test images (0.8510 and 0.8822 in Table 3).

Expected 0.8510 +/- 0.0851 (Rat 1) and 0.8822 +/- 0.0882 (Rat 2), on the images matched above.

In [ ]:
SEM_EXPECTED = {"Rat 1": (0.8510, 0.0851), "Rat 2": (0.8822, 0.0882)}
main = sem[sem.variant == "segment.py"].set_index("sample")
for name, (e, t) in SEM_EXPECTED.items():
    v = main.loc[name, "accuracy"]
    print(f"{name}: accuracy {v:.4f}, expected {e} +/- {t} ({e - t:.4f} to {e + t:.4f}), within: {abs(v - e) <= t}")
for v_ in sem.variant.unique():
    if v_ != "segment.py":
        print(f"variant {v_}: " + ", ".join(f"{r['sample']} {r.accuracy:.4f}" for _, r in sem[sem.variant == v_].iterrows()))
names = ["bg", "myelin", "axon"]
for name, cm in sem_conf.items():
    cm = cm / cm.sum()
    print(f"{name} confusion (fraction of pixels): " + "; ".join(
          f"gt {names[i]} -> " + ", ".join(f"{names[j]} {cm[i, j]:.3f}" for j in range(3)) for i in range(3)))
# Bearing on myelin-dice-below-axon-dice and sensitivity-above-93, whose criteria test the mouse images only.
for name, r in main.iterrows():
    print(f"{name}: myelin Dice {r.myelin_dice:.4f} below axon Dice {r.axon_dice:.4f}: {r.myelin_dice < r.axon_dice}; "
          f"sensitivity {r.sensitivity:.3f} above 0.93: {r.sensitivity > 0.93}")


**Result: reproduced, resting on an assumption.** The two images of the AxonDeepSeg SEM dataset that have the pixel size,
field of view and fixation of Table 1's SEM test rows (rat 6, 0.13 um, 150 x 97 um, for Rat 1; rat 8,
0.07 um, 108 x 77 um, for Rat 2; six of the other eight match Table 1's training rows in all three, and
rat 2 and rat 7 have the pixel size of the row of three images at 0.05 to 0.17 um) give a pixel-wise
accuracy of 0.8955 (Rat 1) and 0.8757 (Rat 2), within 0.7659 to 0.9361 and 0.7940 to 0.9704. Rat 1's 0.8955 is above the
statement's "between 85% and 88%", though inside the criterion's tolerance. The
rebuilt SEM network agrees with its saved graph (largest difference below 1e-04, printed above, and the same
class on every pixel). With the nearest-neighbour resize back the accuracy is 0.8923 and 0.8937, with the 2025
masks of Rat 1 0.8940, and with the contrast inverted 0.3047 and 0.3220. The data are not the
dandiset's and are not named by the paper; they are tied to it by Table 1 only. The outcome assumes that these two images are the
paper's Rat 1 and Rat 2 test images, which Table 1's pixel size, field of view and fixation support
but do not prove. On both images the
myelin Dice is below the axon Dice and the sensitivity is above 0.93 (0.964 and 0.979), which the
criteria of myelin-dice-below-axon-dice and sensitivity-above-93 do not count, as they test the
mouse images only.


## The macaque image (data elsewhere)

The macaque TEM image of Table 3 is not in the dandiset. The White Matter Microscopy Database,
which the paper's Data availability statement names, holds the macaque corpus callosum TEM
images of ref. 7 (folder MontrealNeurologicalInstitute/histology/Macaque of OSF node yp4qg):
eight samples of three images each, 2300 x 3040 pixels (20.7 x 27.4 um at the 0.009 um of
Table 1, which gives 27 x 21 um2), and one myelin and one axon mask per sample. The paper
names neither the sample nor the image it tested. Each mask is matched here to the image of its
sample that it fits (myelin is the darkest structure, so the grey-level contrast between the
rest of the image and the myelin mask is large for the right image and near 0 for the others),
and every matched image is one reading of "the macaque test image".

### What the deposit says about the macaque images

The deposit's own description, its wiki, and every entry of the macaque folder and its sub-folders,
to see whether any of them names the image or the masks the paper used.

In [ ]:
desc, pages = ads.osf_node_text()
print("node yp4qg description:", desc.replace("\n", " ")[:600])
for name, text in pages:
    print(f"wiki page {name!r}: {len(text)} characters; mentions macaque: {'macaque' in text.lower()}, "
          f"AxonDeepSeg: {'axondeepseg' in text.lower()}")
for path in [ads.MACAQUE_ROOT.rsplit("/", 2)[0], ads.MACAQUE_ROOT.rsplit("/", 1)[0], ads.MACAQUE_ROOT,
             ads.MACAQUE_ROOT + "/data", ads.MACAQUE_ROOT + "/segmentation"]:
    lst = ads.osf_listing(path)
    print(f"{path}: " + ", ".join(f"{r['name']} ({r.kind})" for _, r in lst.iterrows()))


In [ ]:
mfiles = ads.osf_macaque_files()
print(mfiles.to_string())
mac_rows = []
for s in sorted(mfiles["sample"].unique()):
    seg = mfiles[(mfiles.kind == "segmentation") & (mfiles["sample"] == s)].set_index("name").id
    my = ads.read_osf_nifti(seg["myelin.nii"])
    ax = ads.read_osf_nifti(seg["avf.nii"])
    m = my > 0
    m_shift = np.roll(m, (50, 50), axis=(0, 1))
    cands = []
    for _, r in mfiles[(mfiles.kind == "data") & (mfiles["sample"] == s)].iterrows():
        im = ads.read_osf_nifti(r.id)
        cands.append((im[~m].mean() - im[m].mean(), im[~m_shift].mean() - im[m_shift].mean(), r["name"], im))
    for c, cs, name, im in cands:
        print(f"{s} {name}: shape {im.shape}, grey {im.min():.0f} to {im.max():.0f}, integer valued {bool(np.all(im == np.round(im)))}; "
              f"contrast {c:.1f}, with the mask shifted by 50 px {cs:.1f}")
    print(f"{s} masks: myelin values {np.unique(my).tolist()}, axon values {np.unique(ax).round(5).tolist()}, "
          f"pixels in both {int((m & (ax > 0)).sum())}")
    c, cs, name, im = max(cands, key=lambda t: t[0])
    row = {"sample": s, "image": name, "contrast": c, "contrast_shifted": cs, "matched": c > 20}
    if row["matched"]:
        gt = ads.macaque_ground_truth(my, ax)
        im8 = np.round(im).astype(np.uint8)
        for invert, tag in ((True, "inv"), (False, "noinv")):
            stitched, rs_shape, n_patches = ads.segment_stitched(im8, ads.MACAQUE_PIXEL_UM, weights, config, invert=invert)
            pred = ads.resize_back(stitched, im8.shape)
            row.update({f"{tag}_{k}": v for k, v in ads.scores(pred, gt).items()})
            if invert:
                row["confusion"] = ads.confusion(pred, gt)
                # Variant: nearest-neighbour resize back to the native shape (not the paper's code).
                row.update({f"nn_{k}": v for k, v in ads.scores(ads.resize_back(stitched, im8.shape, order=0), gt).items()})
        row["n_patches"] = n_patches
    mac_rows.append(row)
mac = pd.DataFrame(mac_rows)
print(mac[["sample", "image", "contrast", "contrast_shifted", "matched"]].round(1).to_string())
print("samples whose masks fit one of their images:", int(mac.matched.sum()), "of", len(mac))

## table3-tem-macaque-accuracy

> On the macaque corpus callosum TEM test image the mouse-trained TEM model reaches a pixel-wise
> accuracy of 84% (0.8438).

Expected 0.8438, tolerance 0.0844 (0.7594 to 0.9282). One reading per matched image.

In [ ]:
mm = mac[mac.matched].set_index("sample")
lo_m, hi_m = 0.8438 - 0.0844, 0.8438 + 0.0844
cols = ["image", "inv_accuracy", "inv_axon_dice", "inv_myelin_dice", "inv_sensitivity", "inv_precision", "noinv_accuracy"]
print(mm[cols].round(4).to_string())
within = (mm.inv_accuracy >= lo_m) & (mm.inv_accuracy <= hi_m)
print(f"readings within {lo_m:.4f} to {hi_m:.4f}: {int(within.sum())} of {len(mm)} ({', '.join(mm.index[within])})")
print("accuracy over the readings: %.4f to %.4f, mean %.4f" % (mm.inv_accuracy.min(), mm.inv_accuracy.max(), mm.inv_accuracy.mean()))
print("Table 3, macaque row, for the other measures: axon Dice 0.9069, myelin Dice 0.7519, sensitivity 0.9429, precision 0.8129")
for m in ["axon_dice", "myelin_dice", "sensitivity", "precision"]:
    print(f"  {m}: {mm['inv_' + m].min():.4f} to {mm['inv_' + m].max():.4f}")
print("ground-truth axon objects per image:", mm.inv_P.astype(int).to_dict(), "| patches per image:", mm.n_patches.astype(int).to_dict())
print("Variant, nearest-neighbour resize back: accuracy %.4f to %.4f, axon Dice %.4f to %.4f, myelin Dice %.4f to %.4f" % (
      mm.nn_accuracy.min(), mm.nn_accuracy.max(), mm.nn_axon_dice.min(), mm.nn_axon_dice.max(),
      mm.nn_myelin_dice.min(), mm.nn_myelin_dice.max()))
print("Confusion matrices, fraction of the image's pixels (rows ground truth, columns prediction; bg, myelin, axon):")
names = ["bg", "myelin", "axon"]
for s_, r in mm.iterrows():
    cm = r.confusion / r.confusion.sum()
    print(f"{s_} {r.image}: " + "; ".join(f"gt {names[i]} -> " + ", ".join(f"{names[j]} {cm[i, j]:.3f}" for j in range(3)) for i in range(3)))
tot = sum(mm.confusion.values)
tot = tot / tot.sum()
print("all 7 images pooled: ground-truth myelin predicted as background %.3f, as axon %.3f, as myelin %.3f of all pixels; "
      "ground-truth axon predicted as background %.3f, as myelin %.3f; ground-truth background predicted as axon %.3f, as myelin %.3f" % (
      tot[1, 0], tot[1, 2], tot[1, 1], tot[2, 0], tot[2, 1], tot[0, 2], tot[0, 1]))
print("errors that are ground-truth myelin: %.3f of all misclassified pixels" % ((tot[1, 0] + tot[1, 2]) / (1 - np.trace(tot))))


**Result: blocked (methods underspecified).** The masks of 7 of the 8 macaque samples fit one of
their images; those of sample 4 fit none of its three images. On the 7 matched images the
pixel-wise accuracy ranges from 0.5729 to 0.7634: one reading (sample 3) is inside 0.7594 to
0.9282 and six are below it, so the readings disagree, and neither the paper nor the deposit says
which image was tested (the deposit's description, wiki and folders, printed above, hold no
description of the macaque images). The other measures are below Table 3's macaque row on every
image (axon Dice 0.6268 to 0.8070 against 0.9069, myelin Dice 0.5032 to 0.6481 against 0.7519,
sensitivity 0.289 to 0.606 against 0.9429). The errors are mostly ground-truth myelin: pooled over
the 7 images, 0.146 of all pixels are ground-truth myelin predicted as background and 0.090 ground-truth
myelin predicted as axon, 0.719 of all misclassified pixels, so the masks mark much more myelin than
the model finds on both of its borders; 0.067 of the pixels are ground-truth axon predicted as
background. With a nearest-neighbour resize back the accuracy is 0.5797 to 0.7678, so the resize
step of the code does not explain the gap. The same TEM code, with the same contrast inversion and
resize, reproduces the mouse rows of Table 3 (table3-tem-mice-accuracy and
table3-tem-mice-dice-detection), so these masks, or these images, may not be the ones the paper
scored; this is not shown, so the claim is not marked as not reproduced. Without the inversion the
accuracy is 0.1810 to 0.3540.


## macaque-myelin-underestimated

> The TEM model slightly underestimates the myelin sheaths of the macaque sample compared with
> its ground truth.

The criterion's reading: the predicted myelin area is below the ground-truth myelin area.

In [ ]:
ratio = mm.inv_pred_myelin_frac / mm.inv_gt_myelin_frac
print(pd.DataFrame({"pred_myelin_frac": mm.inv_pred_myelin_frac, "gt_myelin_frac": mm.inv_gt_myelin_frac,
                    "ratio": ratio, "pred_axon_frac": mm.inv_pred_axon_frac, "gt_axon_frac": mm.inv_gt_axon_frac}).round(3).to_string())
print("readings with predicted myelin area below the ground truth:", int((ratio < 1).sum()), "of", len(mm),
      "| ratio %.3f to %.3f" % (ratio.min(), ratio.max()))
mouse_ratio = res.inv_pred_myelin_frac / res.inv_gt_myelin_frac
print("for comparison, mouse images: ratio %.3f to %.3f, median %.3f" % (mouse_ratio.min(), mouse_ratio.max(), mouse_ratio.median()))

**Result: reproduced, resting on an assumption.** On all 7 matched macaque images the predicted myelin area is below the
ground truth's. The underestimate is large: the predicted myelin fraction is 0.352 to 0.512 of
the ground truth's, against 0.788 to 1.266 (median 0.974) on the mouse images, so the paper's
"slightly" does not hold on these images. The criterion tests the direction only; whether it should
also test the size is asked of a reader (questions/zaimi-2018.yaml). The direction comes from the
same images and masks whose other measures are far from Table 3 (table3-tem-macaque-accuracy), so it
carries little weight. The outcome assumes that the masks of ref. 7 in the White Matter Microscopy
Database are the paper's macaque ground truth: the paper's macaque sample and its labels come from
ref. 7 ("More details can be found about the generation of labels for the macaque 7", Methods,
Ground truth labelling) and its Data availability statement names this database, but it does not
say which masks it scored against, and the gap in the other
measures leaves open that they differ. The direction holds on every one of the 7 images, so it does
not depend on which image was tested.


## table3-other-samples-dice-detection

> For the SEM test images and the macaque TEM image the axon Dice, myelin Dice, sensitivity and
> precision are rat 1 0.9089, 0.8193, 0.9699, 0.8468; rat 2 0.9244, 0.8389, 0.9876, 0.7987; human
> 0.8089, 0.7629, 0.9300, 0.7306; macaque 0.9069, 0.7519, 0.9429, 0.8129.

Tolerance 10% of each value. The rat values are those of the matched SEM test images, the macaque
values one reading per matched macaque image; the human image is not public.

In [ ]:
TABLE3_OTHER = {"Rat 1": (0.9089, 0.8193, 0.9699, 0.8468), "Rat 2": (0.9244, 0.8389, 0.9876, 0.7987),
                "Human": (0.8089, 0.7629, 0.9300, 0.7306), "Macaque": (0.9069, 0.7519, 0.9429, 0.8129)}
q = ["axon_dice", "myelin_dice", "sensitivity", "precision"]
n_in = n_tested = 0
for name in ["Rat 1", "Rat 2"]:
    r = main.loc[name]
    for m, e in zip(q, TABLE3_OTHER[name]):
        ok = abs(r[m] - e) <= 0.1 * e
        n_in += ok; n_tested += 1
        print(f"{name} {m:12s} {r[m]:.4f}  expected {e} +/- {0.1 * e:.4f}  within: {ok}")
for m, e in zip(q, TABLE3_OTHER["Macaque"]):
    ok = (abs(mm["inv_" + m] - e) <= 0.1 * e)
    print(f"Macaque {m:12s} {mm['inv_' + m].min():.4f} to {mm['inv_' + m].max():.4f} over the 7 readings  expected {e} +/- {0.1 * e:.4f}  "
          f"readings within: {int(ok.sum())} of {len(mm)}")
print(f"rat values within tolerance: {n_in} of {n_tested}; human values: not tested (image not public)")


**Result: blocked (data missing).** The human SEM test image is not public, so 4 of the 16 values
cannot be tested. The 8 rat values are all within tolerance on the matched SEM images (axon Dice
0.9282 and 0.9186, myelin Dice 0.8720 and 0.8309, sensitivity 0.964 and 0.979, precision 0.887 and
0.801). The 4 macaque values are outside tolerance on every one of the 7 macaque readings, which
rest on images and masks that may not be the paper's (table3-tem-macaque-accuracy).


## Claims not tested

The dandiset holds only the mouse TEM images and their masks. The human SEM test image (Table 3,
row Human; 0.13 um, 715 x 735 um2 in Table 1) and the full rat cervical spinal cord slice of
Figures 6 and 7 are not in it. The White Matter Microscopy Database (OSF node yp4qg), which the paper
names, holds human spinal cord images (PolytechniqueMontreal/Human, one zip file) whose description,
printed below, gives a pixel size of 230 nm, not the 0.13 um of the test image,
and the AxonDeepSeg SEM dataset holds rat images only. The database holds two SEM panoramas of an
adult rat cervical spinal cord, listed below: PolytechniqueMontreal/RatSpinalCordNov2016 (0.095 um,
dated 2016-11-17) and PolytechniqueMontreal/RatSpinalCord2017 (0.13 um, dated 2017-04-27, next to
a map of axon equivalent diameter). Either could be the slice of Figures 6 and 7, but the paper gives
no pixel size, date or deposit for that slice, so neither can be shown to be it, and the Figure 7
claims also leave the tract boundaries and the white matter mask open. They are not run here, and
these claims are not tested.


In [ ]:
# The human spinal cord images and the two SEM panoramas of a rat cervical spinal cord in the
# White Matter Microscopy Database.
for path in ["PolytechniqueMontreal/Human", "PolytechniqueMontreal/RatSpinalCordNov2016/histology", "PolytechniqueMontreal/RatSpinalCord2017/Histology"]:
    lst = ads.osf_listing(path)
    print(f"----- {path}")
    print(lst[["name", "size", "modified"]].to_string())
    d = lst[lst.name == "description_histology.json"]
    if len(d):
        print(ads.osf_text(d.id.iloc[0]))


In [ ]:
print("species in participants.tsv:", sorted(set(pd.read_csv(pd.io.common.BytesIO(
      ads.read_bytes(assets.set_index('path').asset_id['participants.tsv'])), sep='\t').species)))
print("modalities in the file names:", sorted(set(assets.path.str.extract(r'_(TEM|SEM|BF|[A-Z]+)\.png$')[0].dropna())))
print("body part in the sidecars:", sorted({d['BodyPart'] + '/' + d['BodyPartDetails'] for d in sidecars.values()}))

## Summary

| Claim | Outcome |
|---|---|
| osf-group-labels-link (registry check) | reproduced |
| table3-tem-mice-accuracy | reproduced |
| table3-tem-mice-dice-detection | reproduced |
| myelin-dice-below-axon-dice | reproduced on the mouse images |
| sensitivity-above-93 | reproduced on the mouse images |
| table3-tem-macaque-accuracy | blocked, methods underspecified (which image) |
| macaque-myelin-underestimated | reproduced, assumed (the masks of ref. 7 are the paper's ground truth; direction only, the size is not slight) |
| table3-sem-rat-accuracy | reproduced, assumed (the two images of the AxonDeepSeg SEM dataset matched by Table 1 are the paper's test images) |
| table3-sem-human-accuracy | blocked, data missing |
| table3-other-samples-dice-detection | blocked, data missing (human image); rat values within tolerance, macaque values outside |
| full-slice-processing-time, fig7-corticospinal-tract, fig7-larger-axons-periphery, fig7-tract-diameters, fig7-gratio-range | blocked, data missing |

How the analysis differs from the paper's. The paper's network runs in TensorFlow 1, which cannot
be installed with Python 3.13; it is rebuilt here from the released checkpoint, and its output
matches the model's own saved graph to below 1e-04 on the four patches of the first mouse image,
with the same class on every pixel, and the SEM network matches its saved graph to below 1e-04 on
the patches of Rat 1 (the largest differences are printed in the model check and in the SEM
section). The difference changes slightly from run to run, at the level of floating-point
round-off. Resampling uses the settings of scikit-image 0.13; the original lines of the code are
printed, but the original pipeline could not be run end to end. The paper's test set (3 held-out mice) is not named, so the mouse claims
are tested on all mice and on the control mice, both of which include training mice, with every
set of 3 mice shown. The dandiset's masks are those of 2025 (fibres separated by a 2-pixel
line), not the masks the paper scored against; filling those lines changes the mouse accuracy by less
than 0.002. The SEM rat test images come from the AxonDeepSeg SEM dataset, matched to Table 1 by
pixel size, field of view and fixation. The macaque images and masks come from the White
Matter Microscopy Database, and which of them the paper tested is not known.

What was found about the dataset. The dandiset holds 158 TEM images of 20 mice (8 per mouse,
6 for mouse 25) with three masks each, and the masks fit their images (myelin contrast median
84.4 grey levels in place against 11.8 when shifted). No image is deposited twice. The pixel size
is 0.00236 um, not the 0.002 um of the paper (the README says so; at 0.002 um the images would be
too small for the model's 512-pixel patches). The files do not say which mice are controls; the
White Matter Microscopy Database does, by mouse number. The archive's summary counts 11 samples
where there are 158 images (sample ids repeat across mice). 21 axonmyelin masks are grey plus
alpha pngs, the rest grey. The test split of the paper is not recorded.